In [1]:
import tensorflow as tf

if tf.config.list_physical_devices('GPU'):
    print("GPU is available!")
    print(tf.config.list_physical_devices('GPU'))
else:
    print("GPU is not available. Please check runtime settings.")

GPU is available!
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import os
import pandas as pd
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.image import resize
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from concurrent.futures import ThreadPoolExecutor
import joblib
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [3]:
# Đường dẫn đến dữ liệu trên Kaggle
base_dir = '/kaggle/input/cs114-all-cars/'
csv_dir = '/kaggle/input/split-data-car/'
save_dir = '/kaggle/working/'  # Thư mục lưu kết quả

# Thay đổi đoạn code tải model
weights_path = '/kaggle/input/pretrained-weights/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5'
model = MobileNetV2(weights=weights_path, include_top=False, input_shape=(224, 224, 3))

# Bản đồ từ tên hiệu xe sang CategoryID
category_map = {
    "Others": 0,
    "Honda": 1,
    "Hyundai": 2,
    "KIA": 3,
    "Mazda": 4,
    "Mitsubishi": 5,
    "Suzuki": 6,
    "Toyota": 7,
    "VinFast": 8
}

In [4]:
# Khởi tạo ImageDataGenerator với ít phép tăng cường hơn
datagen = ImageDataGenerator(
    horizontal_flip=True,  # Lật ngang
    rotation_range=15,     # Xoay ngẫu nhiên trong khoảng -15 đến +15 độ
    width_shift_range=0.1,  # Dịch ngang ngẫu nhiên
    height_shift_range=0.1, # Dịch dọc ngẫu nhiên
    brightness_range=[0.8, 1.2],  # Điều chỉnh độ sáng
    zoom_range=0.2         # Zoom ngẫu nhiên
)

def augment_image(image_array):
    """
    Hàm tạo ảnh tăng cường từ một mảng ảnh.
    """
    image_batch = np.expand_dims(image_array, axis=0)  # Chuyển thành batch
    augmented_image_batch = datagen.flow(image_batch, batch_size=1)  # Sinh ảnh tăng cường
    augmented_image = next(augmented_image_batch)[0]  # Lấy ảnh đầu tiên
    return augmented_image

def image_generator(df, augment=False):
    """
    Generator sinh ảnh (gốc và tăng cường với 50% số ảnh nếu bật augment).
    """
    for i, row in enumerate(df.iterrows()):
        try:
            image_path = os.path.join(base_dir, row[1]['ImageFullPath'])
            label = category_map[row[1]['ImageFullPath'].split('/')[0]]

            # Load và tiền xử lý ảnh
            image = load_img(image_path)
            image_array = img_to_array(image)  # Chuyển đổi ảnh thành mảng numpy
            image_array = resize(image_array, (224, 224)).numpy()  # Resize ảnh về (224, 224)

            # Chuẩn hóa ảnh gốc
            image_array = image_array / 255.0
            yield image_array, label  # Trả về ảnh gốc và nhãn

            # Tăng cường 50% số ảnh (chỉ với index chẵn)
            if augment and i % 2 == 0:
                augmented_image = augment_image(image_array)
                augmented_image = augmented_image / 255.0
                yield augmented_image, label  # Trả về ảnh tăng cường và nhãn

        except Exception as e:
            print(f"Error loading image {row[1]['ImageFullPath']}: {e}")

def extract_features(df, split_name, augment=False):
    """
    Trích xuất đặc trưng từ ảnh bằng cách sử dụng generator.
    """
    features_file = os.path.join(save_dir, f"Features_{split_name}_Split_{split_index}.npz")

    # Nếu tệp đã tồn tại, tải lại
    if os.path.exists(features_file):
        print(f"Loading features from {features_file}")
        data = np.load(features_file)
        return data['features'], data['labels']

    features, labels = [], []

    # Sử dụng generator để giảm tải bộ nhớ
    for image_array, label in image_generator(df, augment):
        # Trích xuất đặc trưng từ model
        feature = model.predict(np.expand_dims(image_array, axis=0))
        features.append(feature.flatten())
        labels.append(label)

    # Lưu đặc trưng và nhãn
    np.savez(features_file, features=np.array(features), labels=np.array(labels))
    print(f"Features saved to {features_file}")
    return np.array(features), np.array(labels)

In [5]:
split_index=1

train_path = os.path.join(csv_dir, f"CarDataset-Splits-{split_index}-Train.csv")
test_path = os.path.join(csv_dir, f"CarDataset-Splits-{split_index}-Test.csv")

# Đọc dữ liệu Train và Test
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

In [6]:
X_train, y_train = extract_features(train_df, "Train", augment=True)
X_test, y_test = extract_features(test_df, "Test", augment=False)

# Huấn luyện Random Forest với toàn bộ dữ liệu
classifier = RandomForestClassifier()

# Huấn luyện mô hình
classifier.fit(X_train, y_train)

# Đánh giá trên tập test
accuracy = classifier.score(X_test, y_test)
y_pred = classifier.predict(X_test)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

# In kết quả
print(f"Final Accuracy: {accuracy:.4f}")
print(f"Confusion Matrix:\n{conf_matrix}")
print(f"Classification Report:\n{class_report}")

# Lưu kết quả
save_dir = '/kaggle/working/'
result_path = os.path.join(save_dir, "RF_augment_Results.txt")
with open(result_path, 'w') as file:
    file.write(f"Final Accuracy: {accuracy:.4f}\n\n")
    file.write(f"Confusion Matrix:\n{conf_matrix}\n\n")
    file.write(f"Classification Report:\n{class_report}")

# Lưu mô hình cuối cùng
model_path = os.path.join(save_dir, "RF_augment_Model.joblib")
joblib.dump(classifier, model_path)
print(f"Final model saved to {model_path}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━

/usr/local/lib/python3.10/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━